# Eksplorasi & Pengembangan Pipeline Klasifikasi Sampah dengan GPU NVIDIA

Notebook ini berisi pipeline eksplorasi, pengembangan, dan evaluasi model klasifikasi gambar sampah berbasis deep learning, siap dijalankan di lingkungan dengan GPU NVIDIA dan kernel standar. Outline mengikuti best practice dan referensi dari LAI25-SM011_EcoSortAI.ipynb.

---

## 1. Cek Ketersediaan dan Spesifikasi GPU NVIDIA

Pastikan GPU NVIDIA tersedia dan siap digunakan. Cek dengan TensorFlow dan tampilkan spesifikasi detail menggunakan perintah shell.

In [ ]:
import tensorflow as tf
print("GPU tersedia:", tf.config.list_physical_devices('GPU'))

# Cek spesifikasi GPU NVIDIA (jika di Colab/VM Linux)
!nvidia-smi

: 

## 2. Import Library dan Setup Lingkungan

Import seluruh library yang diperlukan untuk deep learning, visualisasi, manipulasi data, suppress warnings, dan setup environment.

In [ ]:
# Suppress warnings
import warnings
warnings.filterwarnings("ignore")

# TensorFlow & Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, GlobalAveragePooling2D,
                                     Dense, Flatten, Dropout, BatchNormalization)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import VGG16

# Visualisasi
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns

# Manipulasi dan Preprocessing Data
import numpy as np
import os
import zipfile
import shutil
import random
import splitfolders
from tqdm import tqdm
from PIL import Image

# Evaluasi Model
from sklearn.metrics import classification_report, confusion_matrix

# Konversi Model (Opsional)
import tensorflow.lite as lite
import tensorflowjs as tfjs

## 3. Eksplorasi Dataset: Struktur, Distribusi, dan Visualisasi

Tampilkan struktur folder dataset, cek jumlah gambar per kelas, dan visualisasikan beberapa sampel gambar dari setiap kelas untuk memastikan data terorganisir dan siap digunakan.

In [ ]:
# Path ke folder dataset utama
base_dir = '/content/dataset_sampah/sampah-klasifikasi-dataset'  # Ubah sesuai lokasi dataset Anda

# Tampilkan struktur folder dataset
for root, dirs, files in os.walk(base_dir):
    print(f'📁 Folder: {root}')
    print(f'📂 Subfolder: {dirs}')
    print(f'📄 Jumlah file: {len(files)}')
    print('-' * 60)

# Cek jumlah gambar per kelas
distribusi = {}
classes = os.listdir(base_dir)
for cls in classes:
    cls_path = os.path.join(base_dir, cls)
    num_files = len(os.listdir(cls_path))
    distribusi[cls] = num_files
    print(f'Jumlah gambar di kelas "{cls}": {num_files}')

# Visualisasi beberapa sampel gambar dari setiap kelas
import random
fig, axes = plt.subplots(len(classes), 5, figsize=(15, 3 * len(classes)))
for i, cls in enumerate(classes):
    cls_path = os.path.join(base_dir, cls)
    imgs = os.listdir(cls_path)
    sample_imgs = random.sample(imgs, 5)
    for j, img_name in enumerate(sample_imgs):
        img_path = os.path.join(cls_path, img_name)
        try:
            img = Image.open(img_path).convert("RGB")
            axes[i, j].imshow(img)
            axes[i, j].axis('off')
            if j == 2:
                axes[i, j].set_title(cls, fontsize=12)
        except Exception as e:
            print(f"Gagal membuka gambar: {img_path} -> {e}")
            axes[i, j].axis('off')
            axes[i, j].text(0.5, 0.5, 'Error', ha='center', va='center', color='red')
plt.tight_layout()
plt.show()

## 4. Analisis Statistik Gambar (Ukuran, Format, File Size)

Analisis format file gambar, distribusi ukuran gambar (width, height), dan distribusi ukuran file (bytes) untuk mendeteksi outlier dan kebutuhan normalisasi.

In [ ]:
# Analisis format file gambar
dari collections import Counter
formats = []
for cls in classes:
    cls_path = os.path.join(base_dir, cls)
    files = os.listdir(cls_path)
    formats += [f.split('.')[-1].lower() for f in files]
format_count = Counter(formats)
print("Format gambar yang ditemukan dan jumlahnya:", format_count)

# Analisis distribusi ukuran gambar (width, height)
widths, heights = [], []
for cls in classes:
    cls_path = os.path.join(base_dir, cls)
    for fname in os.listdir(cls_path):
        fpath = os.path.join(cls_path, fname)
        try:
            img = Image.open(fpath)
            w, h = img.size
            widths.append(w)
            heights.append(h)
        except:
            print(f'Gagal buka gambar: {fpath}')
print(f'Ukuran gambar - Width: min={min(widths)}, max={max(widths)}, rata-rata={np.mean(widths):.2f}')
print(f'Ukuran gambar - Height: min={min(heights)}, max={max(heights)}, rata-rata={np.mean(heights):.2f}')

# Analisis distribusi ukuran file (bytes)
file_sizes = []
for cls in classes:
    cls_path = os.path.join(base_dir, cls)
    for fname in os.listdir(cls_path):
        fpath = os.path.join(cls_path, fname)
        size = os.path.getsize(fpath)
        file_sizes.append(size)
print(f'Ukuran file (bytes): min={min(file_sizes)}, max={max(file_sizes)}, rata-rata={np.mean(file_sizes):.2f}')

## 5. Preprocessing dan Augmentasi Data

Implementasikan pipeline preprocessing dan augmentasi gambar menggunakan ImageDataGenerator, termasuk normalisasi, rotasi, zoom, shift, shear, flip, dan brightness adjustment.

In [ ]:
# Pipeline augmentasi untuk training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    zoom_range=0.3,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3]
)

# Untuk validation & test (hanya rescale)
val_test_datagen = ImageDataGenerator(rescale=1./255)

## 6. Pemisahan Dataset (Train/Val/Test)

Pisahkan dataset ke dalam subset training, validation, dan test menggunakan splitfolders.ratio dengan rasio yang diinginkan.

In [ ]:
# Path ke dataset mentah
original_dataset_dir = base_dir  # Sudah didefinisikan sebelumnya

# Split ke folder train, val, test
splitfolders.ratio(
    original_dataset_dir,
    output="/content/dataset_split",
    seed=42,
    ratio=(0.7, 0.15, 0.15),
    move=False  # False = copy file, True = pindahkan file
)

# Path setelah split
train_dir = '/content/dataset_split/train'
val_dir = '/content/dataset_split/val'
test_dir = '/content/dataset_split/test'

## 7. Pembuatan dan Visualisasi Image Generator

Buat generator untuk train, validation, dan test. Visualisasikan batch sample dari generator untuk memastikan augmentasi dan label berjalan benar.

In [ ]:
# Buat generator
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)
val_generator = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)
test_generator = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

# Visualisasi batch sample dari train_generator
images, labels = next(train_generator)
plt.figure(figsize=(12, 6))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(images[i])
    plt.axis('off')
    label_idx = np.argmax(labels[i])
    plt.title(train_generator.class_indices.keys().__iter__().__next__() if hasattr(train_generator.class_indices, '__iter__') else label_idx)
plt.tight_layout()
plt.show()

## 8. Transfer Learning: Arsitektur Model dan Fine-Tuning

Bangun model transfer learning berbasis VGG16, freeze base model, tambahkan top layers, compile, dan lakukan fine-tuning pada beberapa layer terakhir.

In [ ]:
# 1. Load VGG16 tanpa fully connected layer, freeze semua layer dulu
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False

# 2. Tambah top layers
x = base_model.output
x = Flatten()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(train_generator.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

# 3. Compile model dengan learning rate agak besar untuk training awal
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 4. Callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint("best_vgg16_model.h5", monitor='val_accuracy', save_best_only=True)
]

# 5. Training awal (feature extraction)
history = model.fit(
    train_generator,
    epochs=20,
    validation_data=val_generator,
    callbacks=callbacks
)

# 6. Fine-tuning: unfreeze beberapa layer terakhir VGG16 untuk training lanjut
for layer in base_model.layers[-4:]:
    layer.trainable = True

# 7. Compile ulang dengan learning rate kecil untuk fine-tuning
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 8. Training fine-tuning
history_finetune = model.fit(
    train_generator,
    epochs=30,
    validation_data=val_generator,
    callbacks=callbacks
)

## 9. Pelatihan Model dan Monitoring

Latih model dengan callbacks (EarlyStopping, ModelCheckpoint), monitor metrik akurasi dan loss pada train/validation, serta visualisasikan learning curve.

In [ ]:
# Visualisasi learning curve akurasi dan loss
plt.figure(figsize=(12,5))
plt.plot(history.history['accuracy'], label='Train Acc (Feature Extraction)')
plt.plot(history.history['val_accuracy'], label='Val Acc (Feature Extraction)')
plt.plot(history_finetune.history['accuracy'], label='Train Acc (Fine-Tuning)')
plt.plot(history_finetune.history['val_accuracy'], label='Val Acc (Fine-Tuning)')
plt.title('Learning Curve Akurasi')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

plt.figure(figsize=(12,5))
plt.plot(history.history['loss'], label='Train Loss (Feature Extraction)')
plt.plot(history.history['val_loss'], label='Val Loss (Feature Extraction)')
plt.plot(history_finetune.history['loss'], label='Train Loss (Fine-Tuning)')
plt.plot(history_finetune.history['val_loss'], label='Val Loss (Fine-Tuning)')
plt.title('Learning Curve Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

## 10. Evaluasi Model: Akurasi, Confusion Matrix, Classification Report

Evaluasi model pada test set, tampilkan confusion matrix, classification report, dan analisis performa per kelas.

In [ ]:
# Evaluasi model pada test set
test_loss, test_acc = model.evaluate(test_generator)
print(f"Akurasi pada Test Set: {test_acc * 100:.2f}%")

# Prediksi data test
y_pred_probs = model.predict(test_generator)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_generator.classes
class_names = list(test_generator.class_indices.keys())

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Klasifikasi Sampah')
plt.show()

# Classification Report
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

## 11. Simpan dan Konversi Model untuk Deployment

Simpan model dalam format .keras, SavedModel, TFLite, dan TensorFlow.js. Siapkan file untuk deployment di berbagai platform.

In [ ]:
# 1️⃣ Simpan model untuk Streamlit (.keras format)
model.save('model_sampah_vgg16.keras')
print("✅ Model disimpan sebagai .keras")

# 2️⃣ Simpan model sebagai SavedModel untuk TF.js dan TFLite
model.save('model_sampah_vgg16_savedmodel', save_format='tf')
print("✅ Model diekspor sebagai SavedModel (folder)")

# 3️⃣ Konversi ke TFLite
converter = tf.lite.TFLiteConverter.from_saved_model('model_sampah_vgg16_savedmodel')
tflite_model = converter.convert()
with open('model_sampah_vgg16.tflite', 'wb') as f:
    f.write(tflite_model)
print("✅ Model dikonversi dan disimpan sebagai .tflite")

# 4️⃣ Konversi ke TensorFlow.js
!tensorflowjs_converter --input_format=tf_saved_model \
  --output_format=tfjs_graph_model \
  model_sampah_vgg16_savedmodel \
  model_sampah_vgg16_tfjs
print("✅ Model dikonversi ke TensorFlow.js (folder tfjs)")

## 12. Inferensi Gambar Baru

Implementasikan fungsi prediksi gambar baru, lakukan upload file, preprocessing, prediksi, dan visualisasi hasil prediksi.

In [ ]:
from tensorflow.keras.preprocessing import image

def predict_image(img_path):
    # Muat gambar dan ubah ukuran
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = img_array / 255.0  # Normalisasi sesuai data training
    img_array = np.expand_dims(img_array, axis=0)  # Tambah batch dimension

    # Prediksi
    predictions = model.predict(img_array)
    predicted_class = np.argmax(predictions, axis=1)[0]
    class_label = list(train_generator.class_indices.keys())[predicted_class]

    # Tampilkan hasil
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Predicted: {class_label}")
    plt.show()

    return class_label

# Contoh penggunaan:
# predict_image('path_ke_gambar_baru.jpg')

## 13. Deployment Model sebagai REST API (Flask/FastAPI)

Contoh kode untuk membuat REST API sederhana menggunakan Flask/FastAPI agar model dapat diakses oleh aplikasi lain (web, mobile, IoT, dsb).

---

## 14. Analisis Error Lanjutan: Visualisasi Grad-CAM

Implementasi Grad-CAM untuk visualisasi area penting pada gambar yang mempengaruhi prediksi model, membantu analisis error dan interpretasi model.

---

## 15. Integrasi Model ke Aplikasi Streamlit

Contoh kode integrasi model ke aplikasi Streamlit untuk demo interaktif klasifikasi gambar sampah.

In [ ]:
# Contoh Integrasi Model ke Streamlit
import streamlit as st
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np

st.title('Demo Klasifikasi Sampah dengan VGG16')

uploaded_file = st.file_uploader('Upload gambar sampah', type=['jpg', 'jpeg', 'png'])
if uploaded_file is not None:
    img = image.load_img(uploaded_file, target_size=(224, 224))
    st.image(img, caption='Gambar yang diupload', use_column_width=True)
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    model = load_model('model_sampah_vgg16.keras')
    class_names = ['class1', 'class2', 'class3']  # Ganti sesuai label dataset
    preds = model.predict(img_array)
    pred_class = class_names[np.argmax(preds)]
    st.success(f'Prediksi: {pred_class}')

In [ ]:
# Grad-CAM untuk Visualisasi Error dan Interpretasi Model
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model([model.inputs], [model.get_layer(last_conv_layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

def show_gradcam(img_path, model, last_conv_layer_name='block5_conv3'):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name)
    plt.imshow(img)
    plt.imshow(heatmap, cmap='jet', alpha=0.5)
    plt.axis('off')
    plt.title('Grad-CAM Overlay')
    plt.show()

# Contoh penggunaan:
 # show_gradcam('path_ke_gambar.jpg', model)

In [ ]:
# Contoh 1: REST API dengan Flask
from flask import Flask, request, jsonify
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np
import os

app = Flask(__name__)

# Load model (pastikan path sesuai)
model = load_model('model_sampah_vgg16.keras')
class_names = ['class1', 'class2', 'class3']  # Ganti sesuai label dataset

def prepare_image(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    return img_array

@app.route('/predict', methods=['POST'])
def predict():
    if 'file' not in request.files:
        return jsonify({'error': 'No file uploaded'}), 400
    file = request.files['file']
    file_path = os.path.join('uploads', file.filename)
    file.save(file_path)
    img_array = prepare_image(file_path)
    preds = model.predict(img_array)
    pred_class = class_names[np.argmax(preds)]
    os.remove(file_path)
    return jsonify({'prediction': pred_class})

if __name__ == '__main__':
    os.makedirs('uploads', exist_ok=True)
    app.run(host='0.0.0.0', port=5000, debug=True)

# Contoh 2: REST API dengan FastAPI
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import JSONResponse
import uvicorn

app = FastAPI()

@app.post('/predict')
async def predict(file: UploadFile = File(...)):
    file_path = f'uploads/{file.filename}'
    with open(file_path, 'wb') as f:
        f.write(await file.read())
    img_array = prepare_image(file_path)
    preds = model.predict(img_array)
    pred_class = class_names[np.argmax(preds)]
    os.remove(file_path)
    return JSONResponse({'prediction': pred_class})

# Jalankan dengan: uvicorn nama_file:app --reload